# Lidar autoencoder - harom architektura osszehasonlitasa

Mindharom modell ugyanabbol a nyers CARLA pontfelhobol indul, de MAS UTON
jut el a latensig. A latens megy majd az RL agent observationjebe.

| modell | mit lat a halo | latens |
|---|---|---|
| `point_mae` | nyers pontok, patch-ekre bontva | 128 |
| `bev_ae` | madartavlati (BEV) pszeudo-kep | 128 |
| `graph_ae` | range image + dinamikus graf | 128 |

Mindharom **128 dimenzios latenst** ad, ugyanugy, mint a kamera modellek - igy
osszehasonlithatok, es barmelyik kozvetlenul beköthetö az RL observationbe.
A `graph_ae` sajat latense terbeli (16x16x128 = 32768 ertek), azt egy
linearis reteg viszi le 128-ra a 6. szekcioban.

**Miert nem egy kaptafa mind a harom**

A kamera notebookban mindharom modell ugyanazt a 160x80-as kepet kapta, csak
a halo felepitese kulonbozott. Itt maskepp van: a pontfelho **rendezetlen** -
nincs "elso pont", es framenkent mas a darabszam (~5800-14000). Mindharom
modell mas trukkel csinal ebbol valamit, amivel egy halo dolgozni tud:

- `point_mae` - meghagyja pontfelhonek, es patch-ekre bontja (FPS + kNN)
- `bev_ae` - fentrol nezve racsra teriti, es kepkent kezeli
- `graph_ae` - a szenzor sajat geometriaja szerint range image-be teriti

**A notebook felepitese**

1. Modulok betoltese
2. Adatok betoltese
3. Kozos tanito fuggvenyek
4. `point_mae` tanitasa + gorbek
5. `bev_ae` tanitasa + gorbek
6. `graph_ae` tanitasa + gorbek
7. **Osszehasonlitas** azonos frameken

Minden modell a sajat `lidar/<nev>.ckpt` fajljaba mentodik, es minden tanitas
utan felszabadul a VRAM.

## 1. Modulok betoltese

In [ ]:
import gc
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from lidar.point_mae import PointMAE
from lidar.bev_ae import BEVConvAE
from lidar.graph_ae import LidarAE, DDCONFIG, points_to_range_image

DATA_DIR = "dataset/lidar"
LATENT_DIM = 128          # a point_mae-nek es a bev_ae-nek ugyanaz
EPOCHS = 30

# HANY FRAME. A teljes dataset 54326 frame, de a pontfelho-modellek
# nagysagrendekkel lassabbak a kepeknel (az FPS mintavetel es a voxelizacio
# a szuk keresztmetszet), es a teljes adat float32-ben be sem ferne a RAM-ba.
# Merve: 8000 frame x 4096 pont = 0.4 GB, ami kenyelmes. Ha van idod, vidd fel.
N_FRAMES = 8000

# HANY PONT FRAMENKENT. A nyers felhok 5835-14021 pontot tartalmaznak, de a
# haloknak FIX alak kell. 4096 a p5 percentilis (7981) alatt van, tehat
# minden frameben van ennyi pont - nem kell feltolteni semmit.
N_POINTS = 4096

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
def free_vram(*objects):
    """Modellek/tenzorok eldobasa es a GPU cache uritese.

    MIERT KELL: harom modellt tanitunk egymas utan. A PyTorch nem adja
    vissza magatol a memoriat az operacios rendszernek - a cache-ben tartja
    ujrahasznositasra. Ha nem uritjuk, a masodik/harmadik tanitas
    'CUDA out of memory'-val elszall egy 8 GB-os kartyan.
    """
    for o in objects:
        if hasattr(o, "cpu"):
            o.cpu()
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def vram():
    """Aktualis GPU memoriahasznalat GB-ban."""
    if not torch.cuda.is_available():
        return "n/a"
    return (f"{torch.cuda.memory_allocated() / 1e9:.2f} GB foglalt / "
            f"{torch.cuda.memory_reserved() / 1e9:.2f} GB fenntartva")


print("indulaskor:", vram())

## 2. Adatok betoltese

A `dataset/lidar/` mappa `.npy` fajljai, mindegyik egy `(N, 3)` float32 tomb
(XYZ, meterben). A framek 20 Hz-cel, idorendben keszultek - ezert **keverni
kell** oket, kulonben egy batch ugyanannak a par masodpercnek a majdnem
azonos felhoibol allna, es a gradiens torzitana.

**Egysegesites fix pontszamra.** A halok fix alaku tenzort varnak, a nyers
felhokben viszont framenkent mas a pontok szama. Veletlen mintavetellel
vagunk le `N_POINTS` pontot mindegyikbol. Ez nem informaciovesztes a
szempontunkbol: a lidar amugy is mintaveteli a vilagot, es a modellek
(FPS, voxelizacio) sajat maguk is ritkitanak.

In [ ]:
def load_clouds(data_dir=DATA_DIR, n_frames=N_FRAMES, n_points=N_POINTS):
    """npy fajlok -> (n_frames, n_points, 3) float32 tenzor.

    A fajlokbol EGYENLETESEN valogatunk (nem az elso n_frames-t vesszuk),
    hogy a teljes felvetel benne legyen, ne csak az eleje.
    """
    paths = sorted(glob.glob(os.path.join(data_dir, "*.npy")))
    if n_frames < len(paths):
        idx = np.linspace(0, len(paths) - 1, n_frames).astype(int)
        paths = [paths[i] for i in idx]

    rng = np.random.default_rng(0)
    out = np.empty((len(paths), n_points, 3), dtype=np.float32)
    for i, p in enumerate(tqdm(paths, desc="betoltes", unit="frame")):
        pts = np.load(p)
        # replace=True csak akkor lep be, ha egy frame kisebb a vartnal.
        sel = rng.choice(len(pts), n_points, replace=len(pts) < n_points)
        out[i] = pts[sel]
    return torch.from_numpy(out)


t0 = time.time()
clouds = load_clouds()
print(f"{tuple(clouds.shape)}  {clouds.numel() * 4 / 1e9:.2f} GB  "
      f"({time.time() - t0:.0f} s)")
print(f"x [{clouds[..., 0].min():.1f}, {clouds[..., 0].max():.1f}]  "
      f"y [{clouds[..., 1].min():.1f}, {clouds[..., 1].max():.1f}]  "
      f"z [{clouds[..., 2].min():.1f}, {clouds[..., 2].max():.1f}]")

In [ ]:
# Keveres, majd train/val vagas.
clouds = clouds[torch.randperm(len(clouds))]

n_val = int(len(clouds) * 0.15)
x_val, x_train = clouds[:n_val], clouds[n_val:]

print(f"train {len(x_train)}  val {len(x_val)}")

### Nezzuk meg, mit is tanulunk

Harom nezet ugyanarrol a frame-rol - pont az a harom, amit a harom modell lat.

In [ ]:
def show_bev(ax, pts, title="", s=0.4):
    """Madartavlati szorasdiagram, a szin a magassag."""
    ax.scatter(pts[:, 0], pts[:, 1], c=pts[:, 2], s=s, cmap="viridis",
               vmin=-3, vmax=4, linewidths=0)
    ax.scatter([0], [0], c="red", s=40, marker="^")   # az auto helye
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    if title:
        ax.set_title(title, fontsize=10)


frame = x_train[0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) nyers pontfelho felulnezetbol - ezt latja a point_mae
show_bev(axes[0], frame, f"nyers pontfelho ({len(frame)} pont)\npoint_mae ezt kapja")

# (b) oldalnezet, hogy lassuk a magassagot is
axes[1].scatter(frame[:, 0], frame[:, 2], c=frame[:, 2], s=0.4, cmap="viridis",
                vmin=-3, vmax=4, linewidths=0)
axes[1].set_xlim(-50, 50)
axes[1].set_ylim(-6, 9)
axes[1].set_xlabel("x [m]")
axes[1].set_ylabel("z [m]")
axes[1].set_title("oldalnezet\n(a talaj a suru vizszintes sav)", fontsize=10)

# (c) range image - ezt latja a graph_ae
ri = points_to_range_image(frame)
im = axes[2].imshow(ri[0], cmap="magma", aspect="auto")
axes[2].set_xlabel("azimut (1024 oszlop)")
axes[2].set_ylabel("csatorna (64 sor)")
axes[2].set_title("range image\ngraph_ae ezt kapja", fontsize=10)
plt.colorbar(im, ax=axes[2], fraction=0.03)

plt.tight_layout()
plt.show()

In [ ]:
# A bev_ae pszeudo-kepe. Ez egy TANULT jellemzoterkep (64 csatorna), nem
# nyers adat - egy random inicializalt halon nezzuk meg, hogy lassuk a
# racsstrukturat. Tanitas utan ugyanezt megnezzuk ujra.
_probe = BEVConvAE(latent_dim=LATENT_DIM).eval()
with torch.no_grad():
    _bev = _probe.to_bev(x_train[:1])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Melyik cellaban van egyaltalan pont.
occ = (_bev[0].abs().sum(0) > 0).float().numpy()
axes[0].imshow(occ, cmap="gray_r", origin="lower",
               extent=[-51.2, 51.2, -51.2, 51.2])
axes[0].set_title(f"pillar foglaltsag ({occ.mean() * 100:.1f}% cella)\n"
                  f"racs {_probe.grid_h}x{_probe.grid_w}, 0.8 m cella", fontsize=10)
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")

# Egy tanult csatorna a 64-bol.
axes[1].imshow(_bev[0, 0].numpy(), cmap="viridis", origin="lower",
               extent=[-51.2, 51.2, -51.2, 51.2])
axes[1].set_title("pszeudo-kep, 0. csatorna (tanitas elott)\nbev_ae ezt kapja",
                  fontsize=10)
axes[1].set_xlabel("x [m]")

plt.tight_layout()
plt.show()
free_vram(_probe, _bev)

### Interaktiv 3D nezet

A fenti abrak statikusak: a felulnezet elrejti a magassagot, az oldalnezet a
szelesseget. Egy pontfelhonel viszont pont a terbeli szerkezet a lenyeg -
ezert erdemes korbejarni.

Az alabbi abra **egerrel forgathato es zoomolhato**:

- bal gomb + huzas - forgatas
- gorgo - zoom
- jobb gomb + huzas (vagy ket ujj) - eltolas
- dupla klikk - visszaallas alaphelyzetbe

A szin a magassag (z). Igy latszik, ami a 2D nezetekbol nem: hol van az ut
sikja, hol allnak a falak, hol vannak jarmuvek, es mennyire ritkul a felho
kifele (a lidar sugarai legyezoszeruen nyilnak, ezert tavol egyre nagyobb a
res ket sav kozott).

In [ ]:
def show_3d(pts, title="", max_points=25000, size=1.2, extra=None,
            height=650):
    """Interaktiv 3D pontfelho a notebookban (plotly).

    pts        : (N, 3) numpy tomb vagy torch tenzor
    max_points : ennel tobb pontot leritkitunk - 25k folott a bongeszo
                 akadozni kezd, es ugyis atlathatatlan
    extra      : opcionalis lista tovabbi pontfelhokrol, amiket ra akarunk
                 rajzolni: [(pontok, "nev", "szin", meret), ...]
    """
    pts = pts.cpu().numpy() if hasattr(pts, "cpu") else np.asarray(pts)

    # Ritkitas, hogy a bongeszo ne akadjon meg.
    if len(pts) > max_points:
        sel = np.random.default_rng(0).choice(len(pts), max_points, replace=False)
        pts = pts[sel]

    traces = [go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=dict(size=size, color=pts[:, 2], colorscale="Viridis",
                    cmin=-3, cmax=4, opacity=0.85,
                    colorbar=dict(title="z [m]", thickness=12, len=0.6)),
        name=f"pontfelho ({len(pts)})",
        hovertemplate="x %{x:.1f}<br>y %{y:.1f}<br>z %{z:.1f}<extra></extra>",
    )]

    # Az auto helye: a lidar az origoban van (x=0, z=2.4 a szenzor magassaga,
    # de a pontok mar a szenzorhoz kepest vannak megadva).
    traces.append(go.Scatter3d(
        x=[0], y=[0], z=[0], mode="markers",
        marker=dict(size=6, color="red", symbol="diamond"),
        name="ego (lidar)", hoverinfo="name"))

    for e_pts, e_name, e_color, e_size in (extra or []):
        e_pts = e_pts.cpu().numpy() if hasattr(e_pts, "cpu") else np.asarray(e_pts)
        traces.append(go.Scatter3d(
            x=e_pts[:, 0], y=e_pts[:, 1], z=e_pts[:, 2], mode="markers",
            marker=dict(size=e_size, color=e_color), name=e_name))

    fig = go.Figure(traces)
    fig.update_layout(
        title=title,
        height=height,
        margin=dict(l=0, r=0, t=40 if title else 0, b=0),
        legend=dict(x=0, y=1),
        scene=dict(
            # aspectmode="data": a harom tengely ARANYA a valos meretekbol
            # jon. Enelkul a plotly kockara nyujtja a jelenetet, es a lapos
            # ut-sik magasnak tunne.
            aspectmode="data",
            xaxis_title="x [m]  (elore)",
            yaxis_title="y [m]  (balra)",
            zaxis_title="z [m]  (fel)",
            # Kezdo kameraallas: kicsit hatulrol-felulrol, mint egy
            # kovetokamera - igy egybol felismerheto a jelenet.
            camera=dict(eye=dict(x=-1.4, y=-1.4, z=0.9)),
        ),
    )
    fig.show()


show_3d(x_train[0], title="Nyers CARLA lidar pontfelho - forgasd el!")

## 3. Kozos tanito fuggvenyek

**Itt egy fontos kulonbseg a kamera notebookhoz kepest.** Ott mindharom modell
ugyanazzal az MSE-vel tanult, igy a loss szamok kozvetlenul osszemerhetok
voltak. Itt EZ NEM LEHETSEGES, mert a harom modell mas celt rekonstrual:

| modell | loss | mit mer |
|---|---|---|
| `point_mae` | Chamfer | a maszkolt patch-ek pontjainak tavolsaga, **meter^2** |
| `bev_ae` | MSE | a BEV jellemzoterkep hibaja, **dimenziotlan** |
| `graph_ae` | L1 | a range image hibaja, **[-1, 1] skalan** |

Ezert a nyers loss-okat NEM hasonlitjuk ossze egymassal - csak azt nezzuk,
hogy egy modell tanul-e (csokken-e a sajat gorbeje). A vegen a 7. szekcioban
van egy KOZOS metrika, ami mindharomra ugyanazt meri.

A tanito fuggveny viszont kozos: ugyanaz az optimizer, ugyanaz az early
stopping, ugyanaz a checkpoint formatum.

In [ ]:
def evaluate(model, x_data, batch, prep=None):
    """Atlagos loss a teljes adathalmazon.

    A modellek sajat `loss()` metodusat hivjuk - mindegyik tudja, mi a sajat
    rekonstrukcios celja.
    """
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(x_data), batch):
            xb = x_data[i:i + batch].to(DEVICE)
            if prep is not None:
                xb = prep(xb)
            total += float(model.loss(xb)) * len(xb)
            n += len(xb)
    return total / n


def train_model(model, label, batch, epochs=EPOCHS, lr=1e-3, patience=5,
                prep=None):
    """Tanitas. A vegen a LEGJOBB val loss-hoz tartozo sulyok maradnak.

    prep: opcionalis atalakito a nyers pontfelhorol arra, amit a modell var
          (a graph_ae-nek range image kell, a masik kettonek nem kell semmi).

    A visszaadott dict tartalmazza a gorbeket es a ckpt utvonalat, de a
    MODELLT NEM - az a tanitas vegen lekerul a GPU-rol es felszabadul.
    Az osszehasonlitashoz a checkpointbol toltjuk vissza.
    """
    model = model.to(DEVICE)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    hist = {"label": label, "arch": type(model).__name__,
            "train": [], "val": [], "best": float("inf"),
            "params": sum(p.numel() for p in model.parameters()),
            "ckpt": f"lidar/{label}.ckpt"}
    best_state, bad = None, 0

    print(f"{label}  ({hist['params'] / 1e6:.1f}M parameter, lr={lr}, batch={batch})")
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        run, n = 0.0, 0
        perm = torch.randperm(len(x_train))
        bar = tqdm(range(0, len(x_train), batch), desc=f"epoch {ep}/{epochs}",
                   leave=False)
        for i in bar:
            xb = x_train[perm[i:i + batch]].to(DEVICE)
            if prep is not None:
                xb = prep(xb)
            loss = model.loss(xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            run += float(loss) * len(xb)
            n += len(xb)
            bar.set_postfix(loss=f"{float(loss):.5f}")

        tr = run / n
        va = evaluate(model, x_val, batch, prep)
        hist["train"].append(tr)
        hist["val"].append(va)

        # Early stopping: ha `patience` epochon at nem javul a val loss,
        # megallunk. A tullanulas ellen ved es idot sporol.
        if va < hist["best"]:
            hist["best"], bad = va, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"  early stop ({patience} epoch javulas nelkul)")
                break

        print(f"  epoch {ep:2d}  train {tr:.5f}  val {va:.5f}"
              f"{'  *' if bad == 0 else ''}")

    model.load_state_dict(best_state)
    hist["epochs_run"] = len(hist["train"])
    hist["time"] = time.time() - t0

    # MENTES sajat nevvel. Az arch is bekerul, hogy visszatoltesnel tudd,
    # melyik architekturarol van szo.
    torch.save({"state_dict": best_state,
                "hparams": (dict(model.hparams) if hasattr(model, "hparams")
                            else {}),
                "arch": hist["arch"],
                "best_val": hist["best"],
                "train": hist["train"], "val": hist["val"]}, hist["ckpt"])

    print(f"  mentve: {hist['ckpt']}  (best val {hist['best']:.5f}, "
          f"{hist['time'] / 60:.1f} perc)")
    return hist, model


def plot_history(hist, ylabel="loss"):
    """Egy modell tanulasi gorbeje."""
    ep = range(1, len(hist["train"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(ep, hist["train"], "-o", ms=3, label="train")
    plt.plot(ep, hist["val"], "-s", ms=3, label="val")
    plt.axhline(hist["best"], ls=":", c="gray",
                label=f"best val {hist['best']:.5f}")
    plt.xlabel("epoch")
    plt.ylabel(ylabel)
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.title(f"{hist['label']}  ({hist['params'] / 1e6:.1f}M parameter)")
    plt.tight_layout()
    plt.show()

## 4. `point_mae`

### Felepites

A pontfelhot **patch-ekre** bontja, ugy, ahogy a ViT egy kepet:

```
(4096, 3) pont
  -> FPS: 64 kozeppont       egyenletesen szorva a terben
  -> kNN: 32 pont mindegyik kore
  -> (64, 32, 3) patch, mindegyik a sajat kozeppontjahoz normalizalva
  -> mini-PointNet -> (64, 192) token
```

Aztan a patch-ek **60%-at eltakarja**, es csak a maradek 40%-ot adja az
encodernek. A decoder feladata a hianyzo patch-ek visszaepitese.

> **Miert a maszkolas?** Egy sima "epitsd vissza a bemenetet" feladatot a
> halo trivialisan megold (eleg atlagolnia). Ha viszont a felho 60%-a
> hianyzik, a kitoltesehez tenyleg erteni kell a geometriat: hogy az ut sik,
> a fal fuggoleges, az auto doboz alaku.

> **Miert Chamfer es nem MSE?** Az MSE az i-edik josolt pontot az i-edik
> valodihoz parositja, de a pontfelhoben nincs sorrend - ugyanaz az alak
> sokfele sorrendben leirhato. A Chamfer minden pontot a hozza legkozelebbihez
> meri, igy sorrendfuggetlen.

In [ ]:
# Merve: batch=32, N=4096 pont mellett a csucs 2.7 GB egy 8 GB-os kartyan.
model = PointMAE(latent_dim=LATENT_DIM, num_group=64, group_size=32)
hist_point_mae, model = train_model(model, label="point_mae", batch=32, lr=1e-3)

plot_history(hist_point_mae, ylabel="Chamfer [m^2]")

In [ ]:
# VRAM felszabaditasa a kovetkezo modell elott.
# A sulyok mar a lidar/point_mae.ckpt fajlban vannak, a modell elengedheto.
free_vram(model)
print("point_mae utan:", vram())

## 5. `bev_ae`

### Felepites

A PointPillars pillar-encodere + egy konvolucios autoencoder:

```
(4096, 3) pont
  -> pillar voxelizacio: 128x128-as racs, 0.8 m-es cella
  -> PillarFeatureNet (mini-PointNet oszloponkent) -> (64, 128, 128) pszeudo-kep
  -> 4 db Conv2d(k=4, s=2) -> (256, 8, 8) -> flatten -> Linear -> 128
```

Innentol ez pontosan ugyanaz, mint a `camera_ae`: egy kep-autoencoder. A
kulonbseg, hogy a "kep" nem RGB, hanem egy **tanult 64 csatornas
jellemzoterkep**, es a rekonstrukcios loss is erre megy.

> **Miert oszlop (pillar) es nem kocka (voxel)?** Mert a vezetesnel a
> magassag kevesbe erdekes, mint a felulnezeti elrendezes. Ha a magassag
> mentén is vagnank, harmadik dimenziot kapnank a racsban, ami
> nagysagrendekkel dragabb - a PointPillars pont azzal lett gyors, hogy ezt
> elhagyta.

In [ ]:
# A bev_ae a legdragabb modell: a 128x128-as pszeudo-kep minden konvolucios
# retege sok aktivaciot tarol a backwardhoz. Merve batch=8 eseten a csucs
# 1.77 GB; batch=32 mar OOM-mal elszall egy 8 GB-os kartyan, ha mas is fut
# rajta. Ha nagyobb kartyad van, batch=16 vagy 32 gyorsabb.
model = BEVConvAE(latent_dim=LATENT_DIM)
hist_bev_ae, model = train_model(model, label="bev_ae", batch=8, lr=1e-3)

plot_history(hist_bev_ae, ylabel="MSE (pszeudo-kep)")

In [ ]:
# VRAM felszabaditasa a kovetkezo modell elott.
# A sulyok mar a lidar/bev_ae.ckpt fajlban vannak, a modell elengedheto.
free_vram(model)
print("bev_ae utan:", vram())

## 6. `graph_ae`

### Felepites

A szenzor sajat geometriaja szerint **range image**-be teriti a felhot: 64 sor
(a lidar 64 csatornaja) x 1024 oszlop (azimut), a pixelertek a tavolsag. Ez
vesztesegmentes atteres, mert pont igy keletkezik az adat a szenzorban.

```
(4096, 3) pont -> (1, 64, 1024) range image
  -> Stem (downsampling) -> (64, 16, 128)
  -> 4 dinamikus graf-reteg (kNN a jellemzoterben, retegenkent ujraszamolva)
  -> (16, 16, 128) = 32768 ertek
  -> flatten -> Linear -> tanh -> 128            <- a szuk keresztmetszet
```

> **Miert kell a szuk keresztmetszet?** A `LidarAE` sajat latense
> 16*128*16 = **32768 ertek**, 256-szorosa a masik ket modellenek. Igy nem
> lehetne osszehasonlitani oket, es az RL-be kotve is bajt okozna: az SB3
> `MultiInputPolicy` NEM normalizal, tehat a 32768 elemu vektor elnyomna a
> tobbi observationt (steer, throttle, waypointok). Ezert a burkolo
> osztalyban jon egy `Linear(32768, 128)`, ugyanaz a recept, mint a
> `camera_ae` conv stackje utan.
>
> **A `graph_ae.py`-hoz nem nyulunk** - a bottleneck a notebookban el.

> **Ket tovabbi kulonbseg:**
> - `lr=1e-4`, nem `1e-3` - ez a modell sajat `configure_optimizers()`
>   beallitasa (`betas=(0.5, 0.9)` mellett).
> - A `Linear` par onmagaban **+8.4M parameter**, tehat a modell 7.8M-rol
>   ~16M-re no. Ez az ara annak, hogy egy 32768 elemu terbeli latenst
>   egyetlen 128 elemu vektorra viszunk le.

In [ ]:
# A range image eloallitasa a CPU-n tortenik, framenkent. Ezert a prep
# fuggveny kicsit lassitja a tanitast - cserebe a graph_ae-nek nem kell
# atirni a bemeneti formatumat.
def to_range_image(points):
    """(B, N, 3) pontfelho -> (B, 1, 64, 1024) range image."""
    imgs = [points_to_range_image(p.cpu().numpy()) for p in points]
    return torch.from_numpy(np.stack(imgs)).to(points.device)


# A LidarAE nem Lightning modul es nincs sajat loss() metodusa - a kozos
# train_model() viszont azt hivja. Egy vekony burkolo megoldja.
#
# A burkolo egyben a SZUK KERESZTMETSZETET is hozza: a LidarAE sajat latense
# (16, 16, 128) = 32768 ertek, ami 256-szorosa a masik ket modellenek. Egy
# Linear par leviszi 128-ra, ugyanugy, ahogy a camera_ae csinalja a conv
# stack utan. Igy mindharom modell azonos meretu latenst ad, es az RL-be is
# kozvetlenul beköthetö.
#
# FONTOS: a graph_ae.py-hoz NEM nyulunk. A bottleneck itt el, a notebookban -
# ha nem valik be, eleg ezt a cellat atirni.
class GraphAEWrapper(torch.nn.Module):
    """LidarAE + 128 dimenzios szuk keresztmetszet + loss().

    latent_dim=None eseten a LidarAE eredeti (16, 16, 128) latense marad.
    """

    def __init__(self, latent_dim=LATENT_DIM, latent_scale=4.0, **kwargs):
        super().__init__()
        self.net = LidarAE(DDCONFIG, **kwargs)
        self.latent_dim = latent_dim
        self.latent_scale = latent_scale

        if latent_dim is not None:
            # Az encoder kimenete (B, z_channels, 16, 128). A 16x128 a DDCONFIG
            # stride-jaibol jon: 64/4 = 16 sor, 1024/8 = 128 oszlop.
            self.spatial = (DDCONFIG["z_channels"], 16, 128)
            flat = self.spatial[0] * self.spatial[1] * self.spatial[2]  # 32768
            self.fc_encode = torch.nn.Linear(flat, latent_dim)
            self.fc_decode = torch.nn.Linear(latent_dim, flat)

    def encode(self, x):
        """Range image -> latens. EZT hasznalja majd az RL."""
        z = self.net.encode(x)
        if self.latent_dim is None:
            return z
        # tanh korlatozza a latenst [-scale, scale] koze, mert az RL
        # observation-space-nek veges hatarai vannak. Ugyanaz a recept,
        # mint a camera_ae-nel es a bev_ae-nel.
        return torch.tanh(self.fc_encode(z.flatten(1))) * self.latent_scale

    def decode(self, z):
        if self.latent_dim is not None:
            z = self.fc_decode(z / self.latent_scale).view(-1, *self.spatial)
        return self.net.decode(z)

    def forward(self, x):
        return self.decode(self.encode(x))

    def loss(self, x):
        # L1 a range image-en - ez a graph_ae sajat docstringjeben ajanlott
        # loss. Az L1 elesebb rekonstrukciot ad, mint az MSE.
        return F.l1_loss(self(x), x)


# A graph_ae a legnagyobb memoriaigenyu modell: a 64x1024-es range image
# dekodere sok aktivaciot tarol a backwardhoz. Merve a csucs batch=4 eseten
# 4.0 GB, batch=8 mar OOM-mal elszall egy 8 GB-os kartyan.
model = GraphAEWrapper()
hist_graph_ae, model = train_model(model, label="graph_ae", batch=4, lr=1e-4,
                                   prep=to_range_image)

plot_history(hist_graph_ae, ylabel="L1 (range image)")

In [ ]:
# VRAM felszabaditasa az osszehasonlitas elott.
# A sulyok mar a lidar/graph_ae.ckpt fajlban vannak, a modell elengedheto.
free_vram(model)
print("graph_ae utan:", vram())

## 7. Osszehasonlitas

Mindharom modell **ugyanazokon a validacios frameken**, amiket egyik sem latott
tanitas kozben. A modelleket a mentett checkpointokbol toltjuk vissza.

Ahogy a 3. szekcioban irtuk, a nyers loss-ok NEM osszemerhetok (mas celt
rekonstrualnak, mas mertekegysegben). Ezert itt ket dolgot nezunk, ami
mindharomra ugyanazt jelenti:

1. **Latens minoseg** - mennyire hasznalja ki a modell a latens teret
   (halott dimenziok, telitettseg), ugyanaz a vizsgalat, mint a kameranal.
2. **Latens simasag** - idoben egymast koveto framek latensei mennyire vannak
   kozel egymashoz. Ez az RL szempontjabol kritikus: ha ket szomszedos frame
   latense nagyot ugrik, az agent zajt lat allapot helyett.

In [ ]:
def load_model(hist):
    """Modell visszatoltese a checkpointbol."""
    ck = torch.load(hist["ckpt"], map_location="cpu", weights_only=False)
    arch = ck["arch"]
    if arch == "PointMAE":
        m = PointMAE(**ck["hparams"])
    elif arch == "BEVConvAE":
        m = BEVConvAE(**ck["hparams"])
    else:
        m = GraphAEWrapper()
    m.load_state_dict(ck["state_dict"])
    return m.eval()


HISTORIES = [hist_point_mae, hist_bev_ae, hist_graph_ae]

# Melyik modell mit var bemenetkent, es mekkora batch fer be neki a GPU-ra.
PREP = {"point_mae": None, "bev_ae": None, "graph_ae": to_range_image}
EVAL_BATCH = {"point_mae": 16, "bev_ae": 8, "graph_ae": 4}

print("Mentett checkpointok:")
for h in HISTORIES:
    size = os.path.getsize(h["ckpt"]) / 1e6
    print(f"  {h['ckpt']:28s} {size:6.1f} MB   best val {h['best']:.5f}")

In [ ]:
# --- Tanulasi gorbek egymas mellett ---
# KULON abran, mert mas a mertekegysegük - egy abrara rakva az egyik gorbe
# lelapulna a masik mellett, es nem latszana, tanul-e egyaltalan.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
units = ["Chamfer [m^2]", "MSE (pszeudo-kep)", "L1 (range image)"]

for ax, h, unit in zip(axes, HISTORIES, units):
    ep = range(1, len(h["train"]) + 1)
    ax.plot(ep, h["train"], "-o", ms=3, label="train")
    ax.plot(ep, h["val"], "-s", ms=3, label="val")
    ax.set_xlabel("epoch")
    ax.set_ylabel(unit)
    ax.set_yscale("log")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title(f"{h['label']}  ({h['params'] / 1e6:.1f}M param)", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Szamszeru osszefoglalo ---
# A 'val loss' oszlop NEM osszehasonlithato a sorok kozott (mas metrika),
# csak azt mutatja, hova jutott az adott modell a sajat celfuggvenyen.

print(f"{'modell':12s} {'val loss':>10s} {'metrika':>18s} {'latens':>9s} "
      f"{'param':>8s} {'epoch':>6s} {'perc':>6s}")
print("-" * 78)

for h, unit in zip(HISTORIES, units):
    print(f"{h['label']:12s} {h['best']:10.5f} {unit:>18s} "
          f"{LATENT_DIM:9d} {h['params'] / 1e6:7.1f}M "
          f"{h['epochs_run']:6d} {h['time'] / 60:6.1f}")

print(f"\nMindharom latens {LATENT_DIM} dimenzios, tehat barmelyik kozvetlenul")
print("beköthetö az RL observationbe.")

In [ ]:
# --- Rekonstrukciok: mit epit vissza a ket pontfelho-alapu modell? ---
#
# A point_mae a MASZKOLT patch-eket epiti vissza - a kek pontok a lathato
# resz (amit az encoder latott), a piros a rekonstrualt hianyzo resz.

n = 3
idx = np.random.choice(len(x_val), n, replace=False)
sample = x_val[idx]

m = load_model(hist_point_mae).to(DEVICE)
with torch.no_grad():
    rebuild, gt, mask, center = m(sample.to(DEVICE))
free_vram(m)

rebuild, gt, mask, center = (t.cpu() for t in (rebuild, gt, mask, center))

fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5.5))
for j in range(n):
    vis_c = center[j][~mask[j]]
    mask_c = center[j][mask[j]]
    # A patch-ek lokalis koordinatait vissza kell tolni a kozeppontjukra.
    rec_pts = (rebuild[j] + mask_c.unsqueeze(1)).reshape(-1, 3)

    ax = axes[j]
    ax.scatter(sample[j][:, 0], sample[j][:, 1], s=0.3, c="lightgray",
               linewidths=0, label="teljes felho")
    ax.scatter(vis_c[:, 0], vis_c[:, 1], s=45, c="tab:blue", marker="o",
               label=f"lathato kozeppont ({len(vis_c)})")
    ax.scatter(rec_pts[:, 0], rec_pts[:, 1], s=1.5, c="tab:red", linewidths=0,
               label=f"rekonstrualt ({len(mask_c)} patch)")
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    if j == 0:
        ax.set_ylabel("y [m]")
        ax.legend(fontsize=8, loc="upper right", markerscale=2)
    ax.set_title(f"point_mae rekonstrukcio #{j}", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Ugyanaz 3D-ben, interaktivan ---
#
# A felulnezeten nem latszik, hogy a rekonstruktalt pontok a helyes
# MAGASSAGBAN vannak-e. Itt korbe lehet jarni: ha a piros pontok a szurke
# felho felszinen ulnek, a modell megtanulta a geometriat; ha a levegoben
# lebegnek vagy a foldbe fulnak, nem.

j = 0                                    # melyik mintat nezzuk a harombol
mask_c_j = center[j][mask[j]]
rec_pts_j = (rebuild[j] + mask_c_j.unsqueeze(1)).reshape(-1, 3)
gt_pts_j = (gt[j] + mask_c_j.unsqueeze(1)).reshape(-1, 3)

show_3d(
    sample[j],
    title=f"point_mae rekonstrukcio 3D  -  {len(mask_c_j)} maszkolt patch",
    extra=[
        (gt_pts_j, f"valodi (maszkolt) pontok", "limegreen", 2.0),
        (rec_pts_j, f"rekonstrualt pontok", "red", 2.0),
        (center[j][~mask[j]], "lathato patch-kozeppontok", "royalblue", 4.0),
    ],
)

In [ ]:
# --- bev_ae: a pszeudo-kep es a rekonstrukcioja ---
m = load_model(hist_bev_ae).to(DEVICE)
with torch.no_grad():
    bev = m.to_bev(sample.to(DEVICE))
    rec = m.decode(m.encode_bev(bev))
bev, rec = bev.cpu(), rec.cpu()
free_vram(m)

fig, axes = plt.subplots(2, n, figsize=(5 * n, 9))
for j in range(n):
    # A 64 csatorna atlaga - egy csatorna onmagaban nehezen olvashato.
    axes[0, j].imshow(bev[j].mean(0), cmap="viridis", origin="lower",
                      extent=[-51.2, 51.2, -51.2, 51.2])
    axes[0, j].set_title(f"eredeti pszeudo-kep #{j}", fontsize=10)
    axes[1, j].imshow(rec[j].mean(0), cmap="viridis", origin="lower",
                      extent=[-51.2, 51.2, -51.2, 51.2])
    axes[1, j].set_title(f"rekonstrukcio  (MSE {F.mse_loss(rec[j], bev[j]):.4f})",
                         fontsize=10)
    for i in (0, 1):
        axes[i, j].set_xlabel("x [m]")
axes[0, 0].set_ylabel("y [m]")
axes[1, 0].set_ylabel("y [m]")

plt.tight_layout()
plt.show()

In [ ]:
# --- graph_ae: a range image es a rekonstrukcioja ---
m = load_model(hist_graph_ae).to(DEVICE)
ri_in = to_range_image(sample.to(DEVICE))
# Framenkent, mert a dekoder memoriaigenyes (lasd a 6. szekcio megjegyzeset).
with torch.no_grad():
    ri_rec = torch.cat([m(ri_in[i:i + 1]) for i in range(len(ri_in))])
ri_in, ri_rec = ri_in.cpu(), ri_rec.cpu()
free_vram(m)

fig, axes = plt.subplots(2 * n, 1, figsize=(14, 2.4 * 2 * n))
for j in range(n):
    axes[2 * j].imshow(ri_in[j, 0], cmap="magma", aspect="auto", vmin=-1, vmax=1)
    axes[2 * j].set_title(f"eredeti range image #{j}", fontsize=10)
    axes[2 * j].set_ylabel("csatorna")
    axes[2 * j + 1].imshow(ri_rec[j, 0], cmap="magma", aspect="auto", vmin=-1, vmax=1)
    axes[2 * j + 1].set_title(
        f"rekonstrukcio  (L1 {F.l1_loss(ri_rec[j], ri_in[j]):.4f})", fontsize=10)
    axes[2 * j + 1].set_ylabel("csatorna")
axes[-1].set_xlabel("azimut")

plt.tight_layout()
plt.show()

In [ ]:
# --- Kihasznaljak-e a modellek a latens teret? ---
#
# halott dimenzio : akinek a szorasa ~0, az nem hordoz informaciot
# telitett        : a tanh hataran (+-latent_scale) ulo ertekek - ezek
#                   gradiense majdnem nulla, tehat mar nem tanulnak
#
# Mindharom modell 128 dimenzios, tanh-hal korlatozott latenst ad, ezert
# ugyanaz a vizsgalat mukodik mindharmon - pont ugy, mint a kameranal.

n_sample = min(500, len(x_val))
sidx = np.random.choice(len(x_val), n_sample, replace=False)
probe = x_val[sidx]

fig, axes = plt.subplots(1, len(HISTORIES), figsize=(6 * len(HISTORIES), 3.6))

for k, h in enumerate(HISTORIES):
    m = load_model(h).to(DEVICE)
    prep = PREP[h["label"]]
    eb = EVAL_BATCH[h["label"]]

    zs = []
    with torch.no_grad():
        for i in range(0, len(probe), eb):
            xb = probe[i:i + eb].to(DEVICE)
            if prep is not None:
                xb = prep(xb)
            zs.append(m.encode(xb).flatten(1).cpu())
    z = torch.cat(zs).numpy()
    # A Lightning modellek hparams-ban tartjak, a wrapper sima attributumban.
    scale = (m.hparams.latent_scale if hasattr(m, "hparams")
             else m.latent_scale)
    free_vram(m)

    std = z.std(axis=0)
    dead = int((std < 0.01 * scale).sum())
    sat = 100 * (np.abs(z) > 0.97 * scale).mean()

    ax = axes[k]
    ax.bar(range(len(std)), np.sort(std)[::-1], width=1.0)
    ax.set_title(f"{h['label']}  ({len(std)} dim)\n"
                 f"{dead} halott dim, {sat:.1f}% telitett", fontsize=10)
    ax.set_xlabel("latens dimenzio (szoras szerint rendezve)")
    ax.set_ylabel("szoras")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("VRAM:", vram())

In [ ]:
# --- Latens simasag: idoben szomszedos framek ---
#
# MIERT SZAMIT: az RL agent a latenst allapotkent latja. Ha ket egymast koveto
# frame (0.05 s kulonbseg) latense nagyot ugrik, az agent zajt lat, nem
# allapotot - a tanulas instabil lesz.
#
# A mero: a szomszedos framek latens-tavolsaga, elosztva a VELETLENSZERUEN
# valasztott parok tavolsagaval. Minel kisebb, annal simabb.
#   ~0.0 : tokeletesen sima
#   ~1.0 : a szomszedos frame ugyanolyan tavol van, mint egy veletlen - a
#          latens gyakorlatilag zaj

# Idorendben egymast koveto framek kellenek, ezert UJRA betoltjuk oket
# keveretlenul, egy rovid szakaszt.
paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.npy")))[:256]
rng = np.random.default_rng(0)
seq = np.empty((len(paths), N_POINTS, 3), dtype=np.float32)
for i, p in enumerate(paths):
    pts = np.load(p)
    seq[i] = pts[rng.choice(len(pts), N_POINTS, replace=len(pts) < N_POINTS)]
seq = torch.from_numpy(seq)

print(f"{'modell':12s} {'szomszed tav':>14s} {'veletlen tav':>14s} {'arany':>8s}")
print("-" * 52)

smooth = {}
for h in HISTORIES:
    m = load_model(h).to(DEVICE)
    prep = PREP[h["label"]]
    eb = EVAL_BATCH[h["label"]]

    zs = []
    with torch.no_grad():
        for i in range(0, len(seq), eb):
            xb = seq[i:i + eb].to(DEVICE)
            if prep is not None:
                xb = prep(xb)
            zs.append(m.encode(xb).flatten(1).cpu())
    z = torch.cat(zs)
    free_vram(m)

    # Szomszedos framek tavolsaga.
    d_neigh = float((z[1:] - z[:-1]).norm(dim=1).mean())
    # Veletlen parok tavolsaga - ez a "skala", amihez viszonyitunk.
    perm = torch.randperm(len(z))
    d_rand = float((z[perm] - z).norm(dim=1).mean())
    ratio = d_neigh / d_rand
    smooth[h["label"]] = ratio

    print(f"{h['label']:12s} {d_neigh:14.3f} {d_rand:14.3f} {ratio:8.3f}")

best = min(smooth, key=smooth.get)
print(f"\nLEGSIMABB latens: {best}  (arany {smooth[best]:.3f})")

# Ertelmezes. Ha MINDEN modell 0.9 folott van, akkor egyik latens sem sima -
# ilyenkor a rangsor nem sokat jelent, mert csak zajt rangsorol.
if min(smooth.values()) > 0.9:
    print("\nFIGYELEM: mindharom arany 0.9 folott van, tehat egyik latens sem")
    print("simabb annal, mintha veletlen frameket parositanank. Ez alultanitas")
    print("jele - fusson tobb epoch, vagy tobb frame (N_FRAMES).")

### Osszegzes

Amit erdemes egyutt nezni a valasztasnal:

- **latens simasag** - az utolso tablazat aranya. Ez a legfontosabb szam az
  RL szempontjabol: alacsony arany = stabil allapotjelzes.
- **halott dimenziok** - ha sok van, a latens tenyleges merete kisebb a
  nevlegesnel, tehat a modell nem hasznalja ki a helyet.
- **tanitasi ido / parameterszam** - kisebb modell gyorsabb az RL futasban is,
  hiszen minden lepesben lefut az encoder.

A nyers `val loss` oszlopot NE hasonlitsd ossze a sorok kozott - mas celt
rekonstrualnak, mas mertekegysegben.

A gyoztes checkpointjat kell beallitani a `config.py`-ban. A latens mindharom
esetben 128 dimenzios, ugyanaz, mint a kamera modelleknel.

> A `graph_ae` visszatoltesehez a `GraphAEWrapper` osztaly kell (a 6.
> szekcioban van definialva), mert a szuk keresztmetszet sulyai abban
> elnek, nem a `graph_ae.py`-ban. Az RL kodban is ezt kell majd behuzni.